Importing libraries

In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import gradio as gr


In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Types of prompts

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [3]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a short, helpful summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [4]:
# Define our user prompt

user_prompt_prefix = """
        Explain the following clearly.

        If it's code:
        - Explain what it does
        - Explain line by line
        - Mention key concepts

        If it's a topic:
        - Explain simply
        - Give examples
        """

In [5]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)


In [7]:
def explain_stream(user_input):
    messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_prefix + user_input}
        ]
    gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
    stream = gemini.chat.completions.create(
            model="gemini-3-flash-preview",
            messages=messages,
            stream=True
        )

    result = ""

    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        result += content
        yield result  

In [8]:
input_box = gr.Textbox(
    lines=10,
    label="Enter Code / Topic / Content"
)

output_box = gr.Markdown(label="Explanation")

view = gr.Interface(
    fn=explain_stream,   
    inputs=input_box,    
    outputs=output_box,
    title="Explain Anything",
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.
